# AI Agents:

- AI Agents
  - Tool Calls
  - Agentic Loop
  - Response API
- ReAct Agent Pattern

## Introduction:

- An AI agent is an autonomous software system that uses artificial intelligence to perceive its environment, make decisions, and take multi-step actions to achieve a specific goal.
- An AI agent can interact with existing system by calling functions. These functions which are made available to an LLM are called tools and the process is called tool calling.
  - LLM decides when to call the tool based on it's intelligence and the received prompt.
  - For every action for a business use case there is probably a tool which the LLM can use to perform that action.
    - Like initiate refund tool
    - Escalate the ticket.
  - We need to tell LLM which tools are available before it can decide which one to use.
  - A tool is not a LLM functionality, LLM just calls tools which contain business logic - CRUD operations.
  - When a tool call is to be made - LLM let's us know about it and we trigger it. (LLM does not automatically call it)
    - It also tells us with what parameter the tool needs to be called.
    - We trigger the tool and get the response.
    - The response is then again fed into LLM using array of messages or whatever context we are preparing.


#### [Tool Schema](https://developers.openai.com/api/docs/guides/function-calling?api-mode=responses):

```json
[{
        "type": "function",
        "name": "get_horoscope",
        "strict": True,
        "description": "Get today's horoscope for an astrological sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "An astrological sign like Taurus or Aquarius",
                },
            },
            "required": ["sign"],
            "additionalProperties": False,
        },
}]
```

| Field | Description |
| ---- | ---- |
|type |	This should always be function |
| name |	The function’s name (for example, get_weather) |
| description |	Details on when and how to use the function |
| parameters |	JSON schema defining the function’s input arguments |
| strict |	Whether to enforce strict mode for the function call |

#### Handling Tool Calls:

When the model calls a function, you must execute it and return the result. Since model responses can include zero, one, or multiple calls, it is best practice to assume there are several.

The response output array contains an entry with the type having a value of `function_call`. Each entry with a `call_id` (used later to submit the function result), `name`, and `JSON-encoded` arguments.

##### LLM Output when a tool call needs to be made:

```json
[
    {
        "id": "fc_12345xyz",
        "call_id": "call_12345xyz",
        "type": "function_call",
        "name": "get_weather",
        "arguments": "{\"location\":\"Paris, France\"}"
    },
    {
        "id": "fc_67890abc",
        "call_id": "call_67890abc",
        "type": "function_call",
        "name": "get_weather",
        "arguments": "{\"location\":\"Bogotá, Colombia\"}"
    },
    {
        "id": "fc_99999def",
        "call_id": "call_99999def",
        "type": "function_call",
        "name": "send_email",
        "arguments": "{\"to\":\"bob@email.com\",\"body\":\"Hi bob\"}"
    }
]
```

##### Executing tools based on name:

```python
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)
    if name == "send_email":
        return send_email(**args)
    raise ValueError(f"Unknown function: {name}")
```

##### Passing the value back to LLM:

```python
input_messages += response.output

for tool_call in response.output:
    if tool_call.type != "function_call":
        continue

    name = tool_call.name
    args = json.loads(tool_call.arguments)

    result = call_function(name, args)
    input_messages.append(
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }
    )
```

## Imports and Dependencies:

In [39]:
import re
import json
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from pydantic import BaseModel, Field, EmailStr, field_validator
from dotenv import load_dotenv
from os import getenv
from enum import Enum
from typing import List, Optional

load_dotenv()
model = "openrouter/free"
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=getenv("OPENROUTER_API_KEY"),
)

def create_chat_message(role, message):
    return {
        "role": role,
        "content":message
    }
    
def create_user_message(message):
    return create_chat_message("user", message)

def create_system_message(message):
    return create_chat_message("system", message)

def create_assistant_message(message):
    return create_chat_message("assistant", message)

def function_call_output_message(call_id, message):
    return {
        "type": "function_call_output",
        "call_id": call_id,
        "output": json.dumps(message)
    }

## Simple Tool Calling:

In [7]:
# Array to store tools :-
tools = []
tools_name_map = {}

In [8]:
# A simple weather look up tool

def weather_lookup(location: str) -> dict:
    """
    Weather look up tool. 
    Should ideally be an API call right now we just return 30 Celcius
    """
    return {
        "temperature": "30",
        "scale": "Celcius",
        "location": "NCR"
    }

# Weather tool schema :-
weather_schema = {
       "type": "function",
        "name": "get_weather",
        "strict": True,
        "description": "Get weather for a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "Location for which we need to get the weather data",
                },
            },
            "required": ["location"],
            "additionalProperties": False,
        }, 
}

tools.append(weather_schema)
tools_name_map["get_weather"] = weather_lookup

In [18]:
def SinglePrompt(messages, tools=[]):
    response = client.responses.create(
        model=model,
        input=messages,
        tools=tools
    )
    return response

In [20]:
llm_response = SinglePrompt([create_user_message("Get the weather of NCR region")], tools)

**Sample response:**
```python
Response(id='gen-1794221423-5XnhDv12ARNXMibWYs55', access_programs=None, created_at=1794221423.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='dots-studio/dots-3-note-preview:free', object='response', output=[ResponseReasoningItem(id='rs_tmp_svqnwpkywi', summary=[], type='reasoning', content=[Content(text='The user wants to get the weather for NCR region. I need to use the get_weather function. The location parameter should be "NCR" or "National Capital Region". I\'ll use "NCR" as it\'s a common abbreviation.', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'), ResponseOutputMessage(id='msg_tmp_habazjv2m5r', content=[ResponseOutputText(annotations=[], text='\n\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None), ResponseFunctionToolCall(arguments='{"location": "NCR"}', call_id='call_0a3ddfdb357b4a2daa33fe98', name='get_weather', type='function_call', id='fc_tmp_6vs4atypzm4', async_=None, caller=None, namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'Location for which we need to get the weather data'}}, 'required': ['location'], 'additionalProperties': False}, strict=True, type='function', allowed_callers=None, async_=None, defer_loading=None, description='Get weather for a location', output_schema=None)], top_p=1.0, background=False, completed_at=1794221434.0, conversation=None, image=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_diagnostics=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier='auto', status='completed', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity=None), tool_usage=None, top_logprobs=0, truncation='disabled', usage=ResponseUsage(input_tokens=194, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0, cache_write_tokens_details=None), output_tokens=78, output_tokens_details=OutputTokensDetails(reasoning_tokens=51), total_tokens=272, cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_input_cost': 0, 'upstream_inference_output_cost': 0}), user=None, presence_penalty=0, frequency_penalty=0, store=False)
```
---

Above contains an output array which can contain the following values :-

| Output Object in Array | `type` Field | Description | Is the LLM Done? (Next Steps) |
| :--- | :--- | :--- | :--- |
| **`ResponseOutputMessage`** | `'message'` | The LLM has generated the final conversational text intended for the user (found inside its nested `content` array). | **Yes.** If this is present and there are no tool calls in the array, the LLM has finished its turn. Display the text. |
| **`ResponseFunctionToolCall`** | `'function_call'` | The LLM is requesting execution of an external function. It provides the tool `name` and the `arguments` to use. | **No.** The LLM is paused and waiting. You must execute this tool in your backend and feed the result back to the LLM. |
| **`ResponseReasoningItem`** | `'reasoning'` | The internal "chain-of-thought" the model performed before taking action (e.g., deciding which tool to use). | **N/A.** This just exposes the model's thinking process. You can log it, render it in a collapsible UI, or ignore it. |
| **`ResponseExecutableCode`** | `'executable_code'` | The LLM generated code (usually Python) intended to be run in a sandboxed environment (Code Interpreter). | **No/Waiting.** If the API runs this automatically, it will pause until execution finishes. If client-side, you must run it and return the output. |
| **`ResponseCodeExecutionResult`** | `'execution_result'` | The console output, variables, or errors resulting from the sandboxed execution of the generated code. | **N/A.** This is usually paired with a follow-up `ResponseOutputMessage` summarizing the results. |
| **`ResponseRefusal`** | `'refusal'` | An explicit block indicating the LLM refused to fulfill the request, typically due to safety or policy violations. | **Yes.** The generation stops here. You should handle the refusal gracefully in your UI. |
| **`ResponseAudio`** | `'audio'` | Multimodal audio generation containing base64-encoded audio data or an audio file buffer. | **Yes.** Render the audio clip for the user in your frontend. |
| **`ResponseImage`** | `'image'` | Multimodal image generation containing base64-encoded image data or a URL. | **Yes.** Render the image in your frontend. |

---

**Output array can contain multiple objects so you have to loop over the output and then check what to perform for each type**

```python
[ResponseReasoningItem(id='rs_tmp_u3elfub2da', summary=[], type='reasoning', content=[Content(text="The user wants weather for the NCR region. I have a get_weather tool that requires a location string. NCR is an abbreviation, likely referring to the National Capital Region of India (Delhi, Gurugram, Noida, etc.). To be helpful, I should call the tool with a location like 'NCR region' or perhaps the National Capital Region. I'll use the location parameter as 'NCR region' since the user asked for that. After getting the result, I'll summarize.", type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),

 
 ResponseOutputMessage(id='msg_tmp_67w3xpd1mkr', content=[ResponseOutputText(annotations=[], text="I'll get the weather information for the NCR region for you.", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None),

 ResponseFunctionToolCall(arguments='{"location": "NCR region"}', call_id='call_ec47d9c1756f4d55bee28328', name='get_weather', type='function_call', id='fc_tmp_cvgl0l6d5ft', async_=None, caller=None, namespace=None, status='completed')]
 ```

 - We can display reasoning on the UI or can ignore it
 - We need to handle the response output message - this means LLM answered something for us. We can display this on UI as well.
 - Last is the tool call which we need to execute and send back the response.
 - The output also gives the chronological order of models thinking / proceeding forward.

----

In chat completions API `response.choices[0].message.tool_calls` array directly told us whether or not a tool call is needed or not but the same is not applicable for responses api.
- The `responses` API is better it shows the chronological thinking. In the classic `chat/completions` API, all text is mashed into one content string, and all tools are mashed into one `tool_calls` array, losing the chronological order of how the model "thought" about the problem.

In [37]:
for output in llm_response.output:
    if output.type == "message":
        print("\n" + output.content[0].text)
    if output.type == "function_call":
        print("\n" + output.name + " " + output.arguments)
        args = json.loads(output.arguments)
        weather = tools_name_map[output.name](args["location"])
        print("\n" + json.dumps(weather))


I'll get the weather information for the NCR region for you.

get_weather {"location": "NCR region"}

{"temperature": "30", "scale": "Celcius", "location": "NCR"}


- We check the output type - message / function call
- If message - display the text as is
- If function call - call the function with required arguments. Since there is only 1 tool, we didn't have to check which tool to call with what parameter.

#### Agentic Loop:

- Above we understood the steps one by one, now we do it in a full loop.

In [45]:
messages = [create_user_message("Get the weather of NCR region")]

while True:
    llm_response = SinglePrompt(messages, tools)
    llmResponseComplete = True
    
    # 1. First, append the assistant's full response to history!
    # (The exact function name depends on your SDK, but you must store the assistant's turn)
    messages.extend(llm_response.output)

    for output in llm_response.output:
        if output.type == "reasoning":
            print("\n" + output.content[0].text)  
            
        elif output.type == "message":
            print("\n" + output.content[0].text)
            
        elif output.type == "function_call":
            args = json.loads(output.arguments)

            # We only have one tool so we don't need to do complex args checking.
            weather = tools_name_map[output.name](args["location"])
            
            # 2. Then, append the tool result matching the call_id
            messages.append(function_call_output_message(output.call_id, weather))
            
            llmResponseComplete = False

    if llmResponseComplete:
        break


The user wants to get the weather for the NCR region. NCR typically refers to the National Capital Region, which includes Delhi, Mumbai, Pune, etc. However, in India context, NCR most commonly refers to the National Capital Region of Delhi. Let me use the get_weather function with "NCR" as the location parameter.

The user asked for the weather of NCR region. I've called the get_weather function with location "NCR" and received a response with temperature 30°C. I should present this information to the user in a clear and friendly manner.



The current weather in NCR is **30°C**.


**Sample Output:**

```markdown
Okay, the user is asking for the weather in the NCR region. Let me see. NCR stands for National Capital Region, which includes Delhi and surrounding areas. The tool available is get_weather, which requires a location parameter. Since the user specified "NCR region," I should use that as the location. I don't need to clarify further because the tool can handle a general location name. I'll call the get_weather function with "NCR region" as the argument.

The function returned weather information for the NCR region. It shows:
- Location: NCR
- Temperature: 30 (in Celsius)
- Scale: Celsius

I should present this information in a clear and helpful way to the user.

The current weather for the NCR (National Capital) region is:

🌡️ **Temperature**: 30°C (Celsius)

That's quite warm! The NCR region is experiencing hot weather conditions right now.
```

**Key Points:**
- We ran the same logic in loop and made sure to only exit the loop when the LLM is done processing the request and not waiting on any tool call or something.
- **Very Important:** `messages.extend(llm_response.output)` without this the Code would run in a loop. Why ? Because LLM would read the first prompt and again ask for a tool call because it would have no context as to whether or not a tool call was asked in previous attempt or not. Adding this we make sure that whatever the assistant has generated is being passed back.

## ReAct Agent Pattern:

The ReAct pattern (Reasoning and Acting) is a foundational design framework for AI agents that combines internal language model reasoning with external tool use in a continuous feedback loop.


The ReAct Loop

1. Thought (Reasoning): The AI uses its internal knowledge to analyze the user's request, track progress, plan the next step, or decide if it needs help from an external tool.
2. Action: The AI invokes a specific external tool (such as a web search engine, a calculator, a database query, or a code interpreter) with required parameters.
3. Observation: The system executes the tool, retrieves the real-world result or data, and feeds that fresh evidence back into the AI.

The agent repeats this Thought → Action → Observation cycle continuously—adjusting its plan if a tool returns an error or unexpected data—until it reaches the final answer or hits a step limit.

Example :-
An agent to fix code bugs might know about the programming language, DB constructs or general information but might not have your code's context so we need it to provide tools to read/edit/create files + any other tool needed to execute the application and test for fixes. (Some IDE already provide these tools). The agent will make changes --> verify --> make edits again --> so on... until the changes are as required.

The loop we wrote earlier is a ReAct loop itself. Most models already contain the reasoning functionality built in them so we only provide the necessary tools to perform the required action. And for models which do not have reasoning we can use Chain Of thought prompting to provide reasoning. The verification can either be done manually or automated via tests which again can be feeded into the LLM thus automating the whole lifecycle.

---

LangChain is a framework which does all of the ReAct pattern internally on its own.